# BioMechAI — PoseC3D Verification & Fine-Tuning Pipeline (Checks 1 to 5)

This notebook verifies and executes the PoseC3D fine-tuning pipeline on GPU (Kaggle/Colab).

## 1. Environment Setup & MMAction2 Installation

In [ ]:
# Install OpenMMLab MMAction2 dependencies
!pip install -q openmim
!mim install -q mmengine
!mim install -q "mmcv>=2.0.0"
!pip install -q "mmaction>=1.0.0"

import torch
print(f"PyTorch: {torch.__version__}, CUDA Available: {torch.cuda.is_available()}")

## 2. Download Pretrained PoseC3D Checkpoint (Kinetics-400)

In [ ]:
# Download official Kinetics-400 PoseC3D checkpoint
!wget -q -c https://download.openmmlab.com/mmaction/skeleton/posec3d/slowonly_r50_u48_240e_ntu60_xsub_keypoint/slowonly_r50_u48_240e_ntu60_xsub_keypoint-7c06c888.pth -O pretrained_posec3d.pth
import os
print(f"Pretrained checkpoint size: {os.path.getsize('pretrained_posec3d.pth') / (1024*1024):.2f} MB")

## 3. CHECK 2: Verify Video-Disjoint Splits on Kaggle

In [ ]:
import pickle
with open('custom_dataset_train.pkl', 'rb') as f:
    train = pickle.load(f)
with open('custom_dataset_val.pkl', 'rb') as f:
    val = pickle.load(f)

train_videos = set((a['label'], a['video_id']) for a in train)
val_videos = set((a['label'], a['video_id']) for a in val)
overlap = train_videos & val_videos

print(f"Train clips: {len(train)}, Train videos: {len(train_videos)}")
print(f"Val clips:   {len(val)}, Val videos:   {len(val_videos)}")
print(f"Overlap (MUST be 0): {len(overlap)}")
assert len(overlap) == 0, "Leakage detected!"

## 4. CHECK 3: Test Model Loading (init_recognizer)

In [ ]:
from mmaction.apis import init_recognizer

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
model = init_recognizer('posec3d_biomechai.py', 'pretrained_posec3d.pth', device=device)
print("Check 3 Passed: Model loaded successfully! Type:", type(model))

## 5. CHECK 4: Tiny End-to-End Dry Run (2 Epochs on 35 Clips)

In [ ]:
# Run tiny dry run
!python -m mmaction.tools.train posec3d_biomechai.py \
    --cfg-options train_dataloader.dataset.ann_file=tiny_train.pkl \
                  val_dataloader.dataset.ann_file=tiny_val.pkl \
                  train_cfg.max_epochs=2 \
                  default_hooks.checkpoint.interval=1 \
    --work-dir ./dryrun_workdir

## 6. CHECK 5: Validate Checkpoint & Resume Ability

In [ ]:
import glob
ckpts = glob.glob('./dryrun_workdir/*.pth')
print("Checkpoints produced in dryrun:", ckpts)
assert len(ckpts) > 0, "No checkpoint found!"

# Test resume from epoch 1
!python -m mmaction.tools.train posec3d_biomechai.py \
    --cfg-options train_dataloader.dataset.ann_file=tiny_train.pkl \
                  val_dataloader.dataset.ann_file=tiny_val.pkl \
                  train_cfg.max_epochs=3 \
    --resume \
    --work-dir ./dryrun_workdir

print("Check 5 Passed: Resume test successful!")

## 7. FULL TRAINING RUN (24 Epochs on All 2,164 Clean Clips)

In [ ]:
# Execute full PoseC3D fine-tuning
!python -m mmaction.tools.train posec3d_biomechai.py \
    --work-dir ./full_training_workdir